In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── 1. Locate files ──────────────────────────────────────────────────────────
input_dir = "/kaggle/input"
csv_files = []
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f.endswith(".csv"):
            csv_files.append(os.path.join(root, f))

print(f"Found {len(csv_files)} CSV file(s)")
for f in csv_files[:5]:
    print(" ", f)

Found 91 CSV file(s)
  /kaggle/input/datasets/ahmedhamedelnaggar/backblaze-2026-data-q2/data_Q2_2026/2026-04-30.csv
  /kaggle/input/datasets/ahmedhamedelnaggar/backblaze-2026-data-q2/data_Q2_2026/2026-05-27.csv
  /kaggle/input/datasets/ahmedhamedelnaggar/backblaze-2026-data-q2/data_Q2_2026/2026-06-18.csv
  /kaggle/input/datasets/ahmedhamedelnaggar/backblaze-2026-data-q2/data_Q2_2026/2026-04-05.csv
  /kaggle/input/datasets/ahmedhamedelnaggar/backblaze-2026-data-q2/data_Q2_2026/2026-04-15.csv


In [3]:
# ── 2b. Load only essential columns first ────────────────────────────────────
from tqdm.notebook import tqdm 
CORE_COLS = [
    "date", "serial_number", "model", "capacity_bytes", "failure",
    # The SMART attributes most predictive of failure (well-known from research)
    "smart_5_raw",    # Reallocated sectors
    "smart_9_raw",    # Power-on hours
    "smart_187_raw",  # Uncorrectable errors
    "smart_188_raw",  # Command timeout
    "smart_197_raw",  # Pending sectors
    "smart_198_raw",  # Uncorrectable sectors
]

dfs = []
for f in tqdm(csv_files):
    # usecols skips unneeded columns at parse time → much faster & lighter
    dfs.append(pd.read_csv(f, usecols=lambda c: c in CORE_COLS, low_memory=False))

df = pd.concat(dfs, ignore_index=True)
print(f"Shape: {df.shape}")
df.head(3)

  0%|          | 0/91 [00:00<?, ?it/s]

Shape: (31968003, 11)


,date,serial_number,model,capacity_bytes,failure,smart_5_raw,smart_9_raw,smart_187_raw,smart_188_raw,smart_197_raw,smart_198_raw
0,2026-04-30,ZYD1V267,ST24000NM002H,24000277250048,0,0.0,9067.0,0.0,0.0,0.0,0.0
1,2026-04-30,ZL004L5R,ST12000NM0008,12000138625024,0,0.0,52077.0,0.0,0.0,0.0,0.0
2,2026-04-30,2GHV2GJV,WDC WUH722222ALE6L4,22000969973760,0,0.0,19464.0,NaN,NaN,0.0,0.0


In [4]:
# ── 3. Basic overview ────────────────────────────────────────────────────────
print("=== dtypes ===")
print(df.dtypes.value_counts())

print("\n=== Column list ===")
print(df.columns.tolist())

=== dtypes ===
float64    6
object     3
int64      2
Name: count, dtype: int64

=== Column list ===
['date', 'serial_number', 'model', 'capacity_bytes', 'failure', 'smart_5_raw', 'smart_9_raw', 'smart_187_raw', 'smart_188_raw', 'smart_197_raw', 'smart_198_raw']


In [6]:
# ── 4. Missing values ────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
missing_df = missing_df[missing_df["missing"] > 0].sort_values("pct", ascending=False)

print(f"{len(missing_df)} columns have nulls\n")
print(missing_df.head(20))

6 columns have nulls

                missing    pct
smart_188_raw  21769603  68.10
smart_187_raw  21750865  68.04
smart_197_raw    773648   2.42
smart_198_raw    237285   0.74
smart_5_raw      208108   0.65
smart_9_raw       39859   0.12


In [7]:
# ── 5. Class balance (the key thing for failure prediction) ──────────────────
print("=== Failure distribution ===")
print(df["failure"].value_counts())
print(f"\nFailure rate: {df['failure'].mean() * 100:.4f}%")
# Backblaze is heavily imbalanced — expect ~0.05–0.1% failure rate

=== Failure distribution ===
failure
0    31966481
1        1522
Name: count, dtype: int64

Failure rate: 0.0048%


In [8]:
# ── 6. SMART column audit ────────────────────────────────────────────────────
# Raw vs normalised columns
raw_cols  = [c for c in df.columns if c.startswith("smart_") and c.endswith("_raw")]
norm_cols = [c for c in df.columns if c.startswith("smart_") and c.endswith("_normalized")]
print(f"SMART raw cols:        {len(raw_cols)}")
print(f"SMART normalised cols: {len(norm_cols)}")

# How many SMART cols are >50% null → candidates for dropping
high_null = missing_df[missing_df["pct"] > 50].index.tolist()
smart_drop = [c for c in high_null if "smart" in c]
print(f"\nSMART cols >50% null (drop candidates): {len(smart_drop)}")

SMART raw cols:        6
SMART normalised cols: 0

SMART cols >50% null (drop candidates): 2
